# 07 - Locked Evaluation and Error Analysis

This notebook evaluates a frozen bundle once on the official labelled Fashionpedia validation split. It does not tune thresholds or models and never renders an unapproved image. A smoke limit validates mechanics; final report numbers require the full desktop bundle and the complete split.

In [5]:
import json,sys
from pathlib import Path
import pandas as pd
cwd=Path.cwd().resolve(); REPO=next((p for p in [cwd,*cwd.parents] if (p/'implementation'/'src').exists()),None)
if REPO is None: raise RuntimeError('Start Jupyter inside the repository.')
sys.path.insert(0,str(REPO/'implementation'/'src'))
from ipcv_attire.dataset_policy import load_policy
from ipcv_attire.evaluation import evaluate_locked_test
from ipcv_attire.pipeline import AttirePipeline
DATA=REPO/'implementation'/'data'; FULL=REPO/'implementation'/'models'/'classical-attire-full'; SMOKE=REPO/'implementation'/'models'/'classical-attire-smoke'
BUNDLE=FULL if (FULL/'manifest.json').exists() else SMOKE
MAXIMUM_IMAGES=None if BUNDLE==FULL else 5
if not (BUNDLE/'manifest.json').exists(): raise FileNotFoundError('Run stage 06 first.')


## Metrics and baselines

Detection: precision, recall and AP50 against boxes; centre-window is the conceptual localization baseline. Segmentation: IoU/Dice against Fashionpedia masks plus rectangular-box-mask IoU. Recognition: per-target F1, balanced accuracy and coverage; report comparisons include majority and HOG-only baselines after the full desktop run. Compliance: macro-F1, balanced accuracy, decided coverage, review rate and confusion matrix.

In [6]:
policy=load_policy(DATA/'dataset-policy.json'); metrics=evaluate_locked_test(AttirePipeline.load(BUNDLE),image_manifest=DATA/'interim'/'manifests'/'fashionpedia-images.csv',fashionpedia_root=DATA/'raw'/'fashionpedia',validation_annotations=DATA/'raw'/'fashionpedia'/'annotations'/'instances_attributes_val2020.json',policy=policy,maximum_images=MAXIMUM_IMAGES)
print('SMOKE MECHANICS ONLY' if MAXIMUM_IMAGES else 'FULL LOCKED TEST'); print(json.dumps(metrics,indent=2))


FULL LOCKED TEST
{
  "evaluated_images": 1158,
  "detection": {
    "precision_at_iou_0_5": 5.4597073596855206e-05,
    "recall_at_iou_0_5": 0.0005271481286241434,
    "ap50": 4.203565476848159e-08,
    "ap_50_to_95": 5.254456846060199e-09,
    "ap_by_iou": {
      "0.50": 4.203565476848159e-08,
      "0.55": 1.0508913692120397e-08,
      "0.60": 0.0,
      "0.65": 0.0,
      "0.70": 0.0,
      "0.75": 0.0,
      "0.80": 0.0,
      "0.85": 0.0,
      "0.90": 0.0,
      "0.95": 0.0
    },
    "true_positive": 2,
    "false_positive": 36630,
    "false_negative": 3792,
    "centre_window_baseline": {
      "precision_at_iou_0_5": 0.0002158894645941278,
      "recall_at_iou_0_5": 0.0002635740643120717
    }
  },
  "segmentation": {
    "mean_iou": 0.0,
    "mean_dice": 0.0,
    "bbox_mask_baseline_mean_iou": 0.5344190123601888,
    "matched_masks": 2
  },
  "recognition": {
    "collared_top": {
      "f1": 0.24083769633507854,
      "balanced_accuracy": 0.5410714285714286,
      "pr_auc"

In [7]:
display(pd.DataFrame(metrics['recognition']).T)
display(pd.DataFrame(metrics['compliance']['confusion_matrix'],index=metrics['compliance']['confusion_matrix_labels'],columns=metrics['compliance']['confusion_matrix_labels']))


,f1,balanced_accuracy,pr_auc,coverage,majority_baseline_f1,hog_only_baseline_f1
collared_top,0.240838,0.541071,0.154483,0.995682,0.000000,0.221954
allowed_sleeve,0.706480,0.547205,0.671837,0.995682,0.762159,0.698489
casual_round_neck_top,0.364602,0.497672,0.248075,0.995682,0.000000,0.373554
revealing_top,0.364855,0.509950,0.227995,0.995682,0.000000,0.360347
formal_bottom_candidate,0.161560,0.486466,0.090383,0.985320,0.000000,0.160410
casual_or_tight_bottom,0.228623,0.486458,0.135709,0.985320,0.000000,0.241998
damaged_bottom,0.049383,0.520367,0.020353,0.985320,0.000000,0.047619
leisurewear,0.007905,0.423067,0.015912,0.985320,0.000000,0.016129
footwear_present,0.000000,0.500000,0.694301,0.000000,0.000000,0.000000
headwear_present,0.000000,0.500000,0.063903,0.000000,0.000000,0.000000


,compliant,non_compliant,review_required
compliant,0,22,0
non_compliant,0,631,0
review_required,0,505,0


## Evidence export and interpretation

Save final full-run metrics under `implementation/outputs/metrics/`. Error figures may be exported only when their image IDs pass the same automatic risk and manual approval gate. Conclusions are limited to Fashionpedia fashion imagery; the system has not been validated on real university photographs.

In [8]:
if MAXIMUM_IMAGES is None:
    output=REPO/'implementation'/'outputs'/'metrics'/'locked-test-metrics.json'; output.parent.mkdir(parents=True,exist_ok=True); output.write_text(json.dumps(metrics,indent=2)+'\n',encoding='utf-8'); print('Saved',output)
else: print('Smoke metrics were not saved as final evidence.')


Saved D:\APU Study\1st Semester\Image Processing and Computer Vision\IPCV Assignment 2\implementation\outputs\metrics\locked-test-metrics.json
